In [2]:
# ============================================================
# India in the Haze – PM2.5 Forecasting (Phase 2)
# Version 3: Lazy Dataset (OOM fix) | Time encoding | 50 epochs
# ============================================================

# ── Cell 1: Imports ──────────────────────────────────────────
import os, gc, math, random, warnings
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs via DataParallel")


# ── Cell 2: Config ───────────────────────────────────────────
class CFG:
    # Paths
    DATA_ROOT  = "/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/raw"
    STATS_PATH = "/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/stats/feat_min_max.mat"
    TEST_DIR   = "/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in"
    OUT_PATH   = "/kaggle/working/preds.npy"
    CKPT       = "/kaggle/working/best_model.pt"

    MONTHS     = ["APRIL_16", "JULY_16", "OCT_16", "DEC_16"]

    INPUT_FEATS = [
        "NH3", "NMVOC_e", "NMVOC_finn", "NOx", "PM25",
        "SO2", "bio", "pblh", "psfc", "q2",
        "rain", "swdown", "t2", "u10", "v10",
    ]
    TARGET_FEAT    = "cpm25"
    WIND_FEATS     = ["u10", "v10"]
    EMISSION_FEATS = ["NH3", "NMVOC_e", "NMVOC_finn", "NOx", "PM25", "SO2", "bio"]

    # Time encoding: sin_hour, cos_hour, sin_month, cos_month  → +4 channels
    N_TIME_CH = 4
    IN_C      = len(INPUT_FEATS) + N_TIME_CH   # 19

    # Temporal
    IN_LEN  = 10
    OUT_LEN = 16
    # ── STRIDE = 3  (was 1 in v2) ──────────────────────────────
    # stride=1 → ~2780 samples → 73 GB peak RAM  →  OOM
    # stride=3 → ~930 samples  →  ~9 GB peak RAM →  safe on 30 GB Kaggle
    # Increase to 4 or 5 if still tight on RAM.
    STRIDE  = 3

    # Training
    BATCH    = 4
    EPOCHS   = 50
    LR       = 3e-4
    WD       = 1e-5
    VAL_FRAC = 0.20

    # Spatial
    H, W = 140, 124


# ── Cell 3: Normalisation helpers ────────────────────────────
def load_stats(path):
    mat = sio.loadmat(path)
    stats = {}
    for key in mat:
        if key.startswith("__"):
            continue
        if key.endswith("_min"):
            feat = key[:-4]
            mn   = float(np.squeeze(mat[key]))
            mx   = float(np.squeeze(mat.get(
                feat + "_max",
                mat.get(key.replace("_min", "_max"), [1.0])
            )))
            stats[feat] = (mn, mx)
    return stats


def normalize(arr, mn, mx, mode="minmax"):
    denom = (mx - mn) if (mx - mn) != 0 else 1.0
    out   = (arr - mn) / denom
    if mode == "sym":
        out = out * 2.0 - 1.0
    return out.astype(np.float32)


def denormalize_cpm25(arr_n, mn, mx):
    denom = (mx - mn) if (mx - mn) != 0 else 1.0
    return arr_n * denom + mn


# ── Cell 4: Time encoding ────────────────────────────────────
def parse_time_channels(time_arr, H, W):
    """
    time_arr : (T,) strings 'YYYY-MM-DDTHH:MM:SS'
    Returns  : (T, H, W, 4)  [sin_h, cos_h, sin_mo, cos_mo]
    Stored as float32 vectors broadcast spatially — tiny memory footprint.
    """
    T = len(time_arr)
    sh  = np.zeros(T, dtype=np.float32)
    ch  = np.zeros(T, dtype=np.float32)
    smo = np.zeros(T, dtype=np.float32)
    cmo = np.zeros(T, dtype=np.float32)

    for i, ts in enumerate(time_arr):
        ts   = str(ts)
        hr   = int(ts[11:13])
        mo   = int(ts[5:7])
        sh[i]  = math.sin(2 * math.pi * hr        / 24)
        ch[i]  = math.cos(2 * math.pi * hr        / 24)
        smo[i] = math.sin(2 * math.pi * (mo - 1)  / 12)
        cmo[i] = math.cos(2 * math.pi * (mo - 1)  / 12)

    def bcast(v):
        return np.broadcast_to(v[:, None, None], (T, H, W)).copy()

    return np.stack([bcast(sh), bcast(ch), bcast(smo), bcast(cmo)], axis=-1)


# ── Cell 5: Load one month (raw arrays, no windowing) ────────
# make_samples() from v2 is REMOVED — windowing is now done lazily
# inside PM25Dataset.__getitem__() to avoid the 73 GB RAM spike.

def load_month(month_dir, stats):
    """
    Loads and normalises one month.
    Returns:
        X : (T, H, W, 19)  — physical feats + time encoding, float32
        y : (T, H, W)      — normalised cpm25, float32
    Memory per month ≈ 0.5 GB  (vs. 9 GB for pre-windowed stride-1 samples)
    """
    feats = []
    for feat in CFG.INPUT_FEATS:
        arr      = np.load(os.path.join(month_dir, feat + ".npy"))
        mn, mx   = stats.get(feat, (float(arr.min()), float(arr.max())))
        if feat in CFG.WIND_FEATS:
            arr_n = normalize(arr, mn, mx, mode="sym")
        elif feat in CFG.EMISSION_FEATS:
            arr_n = np.clip(normalize(arr, mn, mx, mode="minmax"), 0.0, 1.0)
        else:
            arr_n = normalize(arr, mn, mx, mode="minmax")
        feats.append(arr_n)

    X_phys   = np.stack(feats, axis=-1)                             # (T, H, W, 15)
    time_arr = np.load(os.path.join(month_dir, "time.npy"))
    T, H, W  = X_phys.shape[:3]
    X_time   = parse_time_channels(time_arr, H, W)                  # (T, H, W, 4)
    X        = np.concatenate([X_phys, X_time], axis=-1)            # (T, H, W, 19)

    y_raw    = np.load(os.path.join(month_dir, CFG.TARGET_FEAT + ".npy"))
    mn_y, mx_y = stats.get(CFG.TARGET_FEAT,
                            (float(y_raw.min()), float(y_raw.max())))
    y = normalize(y_raw, mn_y, mx_y, mode="minmax")

    return X, y                                                      # raw, un-windowed


# ── Cell 6: [REMOVED] make_samples ───────────────────────────
# Windowing is now done on-the-fly inside PM25Dataset.__getitem__.
# This is the core OOM fix: we never materialise (N, 10, 140, 124, 19)
# all at once — only one (10, 140, 124, 19) slice per training step.


# ── Cell 7: Build month_data list + index list ───────────────
# ── CHANGED from v2 ──────────────────────────────────────────
# v2: accumulated all windows into all_x / all_y → np.array() → OOM
# v3: keep raw month arrays in memory; build a flat list of
#     (month_idx, window_start) tuples. Zero extra RAM for windows.

print("Loading normalisation stats …")
stats = load_stats(CFG.STATS_PATH)

# Ensure cpm25 stats exist (needed for denorm at inference)
if CFG.TARGET_FEAT not in stats:
    print("  cpm25 not in .mat — computing from training data …")
    raw_vals = []
    for month in CFG.MONTHS:
        p = os.path.join(CFG.DATA_ROOT, month, CFG.TARGET_FEAT + ".npy")
        raw_vals.append(np.load(p).ravel())
    cat = np.concatenate(raw_vals)
    stats[CFG.TARGET_FEAT] = (float(cat.min()), float(cat.max()))
    del raw_vals, cat; gc.collect()
    print(f"  cpm25  min={stats['cpm25'][0]:.3f}  max={stats['cpm25'][1]:.3f}")

# Load raw month arrays — small (~0.5 GB each)
month_data = []   # list of (X_month, y_month)
index_list = []   # list of (month_idx, window_start)

window = CFG.IN_LEN + CFG.OUT_LEN   # 26

for m_idx, month in enumerate(CFG.MONTHS):
    month_dir = os.path.join(CFG.DATA_ROOT, month)
    print(f"  Loading {month} …")
    X_m, y_m = load_month(month_dir, stats)
    month_data.append((X_m, y_m))

    T_m = X_m.shape[0]
    starts = list(range(0, T_m - window + 1, CFG.STRIDE))
    for s in starts:
        index_list.append((m_idx, s))

    print(f"    shape X={X_m.shape}  y={y_m.shape}  windows={len(starts)}")

total_samples = len(index_list)
print(f"\nTotal windows: {total_samples}  (stride={CFG.STRIDE})")

# Estimate RAM used
one_month_bytes = month_data[0][0].nbytes + month_data[0][1].nbytes
total_month_bytes = sum(X.nbytes + y.nbytes for X, y in month_data)
print(f"Raw month arrays in RAM: {total_month_bytes / 1e9:.2f} GB  "
      f"(vs ~39 GB for pre-windowed in v2)")


# ── Cell 8: Lazy PM25Dataset ──────────────────────────────────
# ── CHANGED from v2 ──────────────────────────────────────────
# v2: __init__ stored full (N, C, T, H, W) tensor → second full copy → OOM
# v3: __init__ stores only references; __getitem__ slices one window at a time

class PM25Dataset(Dataset):
    """
    Lazy windowing dataset.
    Holds references to raw month arrays + a flat index list.
    __getitem__ extracts exactly one (IN_LEN, H, W, C) slice per call —
    peak extra RAM per worker = one sample ≈ 10 * 140 * 124 * 19 * 4 B ≈ 13 MB.
    """
    def __init__(self, month_data, index_list):
        self.month_data = month_data    # list of (X, y) — shared, not copied
        self.index_list = index_list    # list of (month_idx, start)

    def __len__(self):
        return len(self.index_list)

    def __getitem__(self, idx):
        m_idx, start = self.index_list[idx]
        X_m, y_m    = self.month_data[m_idx]

        x_win = X_m[start           : start + CFG.IN_LEN]             # (10, H, W, 19)
        y_win = y_m[start + CFG.IN_LEN : start + CFG.IN_LEN + CFG.OUT_LEN]  # (16, H, W)

        # (T, H, W, C) → (C, T, H, W)  — contiguous copy, one sample only
        x_t = torch.from_numpy(np.ascontiguousarray(x_win.transpose(3, 0, 1, 2)))
        y_t = torch.from_numpy(np.ascontiguousarray(y_win))

        return x_t, y_t   # (19, 10, 140, 124),  (16, 140, 124)


# Train / val split on index_list only (no data copied)
np.random.shuffle(index_list)
val_size   = int(total_samples * CFG.VAL_FRAC)
train_idx  = index_list[val_size:]
val_idx    = index_list[:val_size]

train_ds = PM25Dataset(month_data, train_idx)
val_ds   = PM25Dataset(month_data, val_idx)

# num_workers=2: each worker holds references to month_data (shared memory via
# fork on Linux), so RAM cost is negligible despite multiple workers.
train_loader = DataLoader(train_ds, batch_size=CFG.BATCH, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG.BATCH, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"Train samples: {len(train_ds)}  |  Val samples: {len(val_ds)}")


# ── Cell 9: Model Definition ──────────────────────────────────
# Unchanged from v2 — DataLoader output shape is identical.

class ConvBnGelu(nn.Module):
    def __init__(self, in_c, out_c, k=3, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, k, padding=p, bias=False),
            nn.BatchNorm2d(out_c),
            nn.GELU(),
        )
    def forward(self, x): return self.block(x)


class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.block = nn.Sequential(ConvBnGelu(in_c, out_c), ConvBnGelu(out_c, out_c))
    def forward(self, x): return self.block(x)


class ConvLSTMCell(nn.Module):
    def __init__(self, in_c, hidden_c, kernel=3):
        super().__init__()
        self.hidden_c = hidden_c
        self.gates    = nn.Conv2d(in_c + hidden_c, 4 * hidden_c,
                                  kernel, padding=kernel // 2)

    def forward(self, x, hc):
        h, c   = hc
        i, f, g, o = self.gates(torch.cat([x, h], dim=1)).chunk(4, dim=1)
        i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
        c_new   = f * c + i * torch.tanh(g)
        return o * torch.tanh(c_new), c_new

    def init_hidden(self, B, H, W, dev):
        z = torch.zeros
        return z(B, self.hidden_c, H, W, device=dev), z(B, self.hidden_c, H, W, device=dev)


class SpatioTemporalUNet(nn.Module):
    def __init__(self, in_c=CFG.IN_C, base=32, lstm_h=64, out_len=CFG.OUT_LEN):
        super().__init__()
        self.out_len = out_len

        self.enc1  = DoubleConv(in_c, base)
        self.down1 = nn.MaxPool2d(2)
        self.enc2  = DoubleConv(base, base * 2)
        self.down2 = nn.MaxPool2d(2)
        self.enc3  = DoubleConv(base * 2, base * 4)
        self.bproj = nn.Conv2d(base * 4, lstm_h, 1)

        self.clstm = ConvLSTMCell(lstm_h, lstm_h)

        self.up2  = nn.ConvTranspose2d(lstm_h,   base * 2, 2, stride=2)
        self.dec2 = DoubleConv(base * 2 + base * 2, base * 2)
        self.up1  = nn.ConvTranspose2d(base * 2, base,     2, stride=2)
        self.dec1 = DoubleConv(base + base, base)
        self.head = nn.Conv2d(base, 1, 1)

    def encode(self, x):
        e1     = self.enc1(x)
        e2     = self.enc2(self.down1(e1))
        bottle = self.bproj(self.enc3(self.down2(e2)))
        return e1, e2, bottle

    def decode(self, h, e1, e2):
        d = F.interpolate(self.up2(h), size=e2.shape[-2:])
        d = self.dec2(torch.cat([d, e2], dim=1))
        d = F.interpolate(self.up1(d), size=e1.shape[-2:])
        d = self.dec1(torch.cat([d, e1], dim=1))
        return self.head(d).squeeze(1)   # (B, H, W)

    def forward(self, x):
        """x : (B, C, T, H, W)  →  (B, out_len, H, W)"""
        B, C, T, H, W = x.shape
        hc = self.clstm.init_hidden(B, H // 4, W // 4, x.device)

        e1_last = e2_last = None
        for t in range(T):
            e1, e2, bottle = self.encode(x[:, :, t])
            hc = (self.clstm(bottle, hc))
            e1_last, e2_last = e1, e2

        preds = []
        for _ in range(self.out_len):
            preds.append(self.decode(hc[0], e1_last, e2_last).unsqueeze(1))
        return torch.cat(preds, dim=1)


_model = SpatioTemporalUNet(in_c=CFG.IN_C, base=32, lstm_h=64, out_len=CFG.OUT_LEN)
if torch.cuda.device_count() > 1:
    _model = nn.DataParallel(_model)
model = _model.to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model params: {n_params / 1e6:.2f}M  |  Input channels: {CFG.IN_C}")


# ── Cell 10: Loss Functions ───────────────────────────────────
def smape_loss(pred, target, eps=1.0):
    return torch.mean(
        2.0 * torch.abs(pred - target) /
        (torch.abs(pred) + torch.abs(target) + eps)
    )


def episode_weighted_loss(pred, target, eps=1.0, epi_weight=3.0):
    base = smape_loss(pred, target, eps)
    B    = target.shape[0]
    flat = target.view(B, -1)
    mu   = flat.mean(1).view(B, 1, 1, 1)
    sig  = flat.std(1).view(B, 1, 1, 1)
    mask = (target > mu + 1.5 * sig).float()
    if mask.sum() > 0:
        return base + epi_weight * smape_loss(pred * mask, target * mask, eps)
    return base


# ── Cell 11: Training Loop ────────────────────────────────────
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WD)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG.EPOCHS, eta_min=1e-6
)
scaler  = GradScaler()
history = []
best_val_loss = float("inf")

print(f"\n{'Epoch':>6}  {'Train':>8}  {'Val':>8}  {'Best':>8}  Status")
print("-" * 56)

for epoch in range(1, CFG.EPOCHS + 1):

    # ── Train ──
    model.train()
    train_loss = 0.0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        with autocast():
            loss = episode_weighted_loss(model(X_b), y_b)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # ── Validate ──
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            with autocast():
                val_loss += smape_loss(model(X_b), y_b).item()
    val_loss /= len(val_loader)

    scheduler.step()

    # ── Best checkpoint (saves every time val improves) ──
    is_best = val_loss < best_val_loss
    if is_best:
        best_val_loss = val_loss
        state = (model.module.state_dict()
                 if hasattr(model, "module") else model.state_dict())
        torch.save({
            "epoch":      epoch,
            "state_dict": state,
            "val_loss":   val_loss,
            "train_loss": train_loss,
        }, CFG.CKPT)
        status = "✓ best saved"
    else:
        status = ""

    history.append({"epoch": epoch, "train": train_loss, "val": val_loss})
    print(f"{epoch:>6}  {train_loss:>8.4f}  {val_loss:>8.4f}  "
          f"{best_val_loss:>8.4f}  {status}")

print(f"\nTraining complete.  Best val SMAPE: {best_val_loss:.4f}  → {CFG.CKPT}")


# ── Cell 12: Load best checkpoint ────────────────────────────
print("\nLoading best checkpoint …")
ckpt = torch.load(CFG.CKPT, map_location=device)
print(f"  Epoch {ckpt['epoch']}  |  val={ckpt['val_loss']:.4f}")

infer_model = SpatioTemporalUNet(
    in_c=CFG.IN_C, base=32, lstm_h=64, out_len=CFG.OUT_LEN
).to(device)
infer_model.load_state_dict(ckpt["state_dict"])
infer_model.eval()


# ── Cell 13: Inference on test data ──────────────────────────
print("Running inference …")

test_feats = []
for feat in CFG.INPUT_FEATS:
    arr      = np.load(os.path.join(CFG.TEST_DIR, feat + ".npy"))  # (218, 10, H, W)
    mn, mx   = stats.get(feat, (float(arr.min()), float(arr.max())))
    if feat in CFG.WIND_FEATS:
        arr_n = normalize(arr, mn, mx, mode="sym")
    elif feat in CFG.EMISSION_FEATS:
        arr_n = np.clip(normalize(arr, mn, mx, mode="minmax"), 0.0, 1.0)
    else:
        arr_n = normalize(arr, mn, mx, mode="minmax")
    test_feats.append(arr_n)

test_X_phys = np.stack(test_feats, axis=-1).astype(np.float32)  # (218, 10, H, W, 15)
N_test, T_test = test_X_phys.shape[:2]

# Time encoding — zeros (safe default). If test timestamps are available,
# load test_in/time.npy and use parse_time_channels() for each sample.
test_X_time = np.zeros(
    (N_test, T_test, CFG.H, CFG.W, CFG.N_TIME_CH), dtype=np.float32
)

test_X   = np.concatenate([test_X_phys, test_X_time], axis=-1)  # (218, 10, H, W, 19)
print(f"  test_X shape: {test_X.shape}")

# (N, T, H, W, C) → (N, C, T, H, W)
test_X_t  = torch.from_numpy(test_X.transpose(0, 4, 1, 2, 3))

all_preds   = []
INFER_BATCH = 8
with torch.no_grad():
    for i in range(0, N_test, INFER_BATCH):
        xb = test_X_t[i : i + INFER_BATCH].to(device)
        with autocast():
            pb = infer_model(xb)           # (B, 16, H, W)
        all_preds.append(pb.cpu().float().numpy())

preds_norm = np.concatenate(all_preds, axis=0)          # (218, 16, H, W)

mn_y, mx_y = stats[CFG.TARGET_FEAT]
preds_phys = denormalize_cpm25(preds_norm, mn_y, mx_y)

preds_out  = preds_phys.transpose(0, 2, 3, 1)           # (218, 140, 124, 16)
print(f"  Final shape: {preds_out.shape}  (expected (218, 140, 124, 16))")
assert preds_out.shape == (218, 140, 124, 16), f"Shape mismatch: {preds_out.shape}"

np.save(CFG.OUT_PATH, preds_out)
print(f"\nSaved → {CFG.OUT_PATH}")


# ── Cell 14: Training history ─────────────────────────────────
print("\n── Training History ──────────────────────────")
print(f"{'Epoch':>6}  {'Train':>8}  {'Val':>8}")
for row in history:
    marker = " ◀ best" if abs(row["val"] - best_val_loss) < 1e-8 else ""
    print(f"{row['epoch']:>6}  {row['train']:>8.4f}  {row['val']:>8.4f}{marker}")


# ============================================================
# ── BONUS: Tips for Leaderboard Improvement ─────────────────
# ============================================================
"""
=== LEADERBOARD BOOST CHECKLIST ===

1. OOM FIX (v3) ✓
   - Lazy dataset: only raw month arrays in RAM (~2 GB total)
   - Windowing done on-the-fly in __getitem__
   - STRIDE=3 gives ~930 samples, safe on 30 GB Kaggle RAM
   - Lower STRIDE for more samples if RAM allows (e.g. STRIDE=2 → ~1400 samples)

2. TIME ENCODING ✓
   - sin/cos hour + sin/cos month added as 4 spatial channels
   - If test_in/time.npy exists, use parse_time_channels() for test too

3. EPISODE-AWARE LOSS ✓
   - epi_weight=3. Try 4–5 if Episode Correlation metric lags.

4. ARCHITECTURE UPGRADES
   - Add Spatial Attention after ConvLSTM hidden state
   - Try PredRNN++ / MIM cell for better long-range temporal coherence
   - Add FNO2D head for long-range spatial correlations

5. LAT/LONG CHANNELS
   - lat_long.npy shape (140, 124, 2). Broadcast over T → IN_C becomes 21.

6. DATA AUGMENTATION
   - Random horizontal flip (physically valid over India E-W axis)
   - Gaussian noise σ=0.01 on inputs

7. ENSEMBLE
   - Train 3 seeds, average predictions → typically -1 to -3% SMAPE

8. POST-PROCESSING
   - Linear calibration (slope + bias) fit on val set predictions

9. LOSS CURRICULUM
   - Epochs 1-10: MSE for stable gradients
   - Epochs 11-50: episode_weighted_loss for peak focus
"""

Device: cuda
Using 2 GPUs via DataParallel
Loading normalisation stats …
  Loading APRIL_16 …
    shape X=(715, 140, 124, 19)  y=(715, 140, 124)  windows=230
  Loading JULY_16 …
    shape X=(739, 140, 124, 19)  y=(739, 140, 124)  windows=238
  Loading OCT_16 …
    shape X=(739, 140, 124, 19)  y=(739, 140, 124)  windows=238
  Loading DEC_16 …
    shape X=(739, 140, 124, 19)  y=(739, 140, 124)  windows=238

Total windows: 944  (stride=3)
Raw month arrays in RAM: 4.07 GB  (vs ~39 GB for pre-windowed in v2)
Train samples: 756  |  Val samples: 188
Model params: 0.76M  |  Input channels: 19

 Epoch     Train       Val      Best  Status
--------------------------------------------------------
     1    0.1234    0.0592    0.0592  ✓ best saved
     2    0.0569    0.0565    0.0565  ✓ best saved
     3    0.0479    0.0302    0.0302  ✓ best saved
     4    0.0427    0.0415    0.0302  
     5    0.0390    0.0259    0.0259  ✓ best saved
     6    0.0353    0.0250    0.0250  ✓ best saved
     7    0

KeyboardInterrupt: 